# DSA4212 small LLM: Kaggle version
Before running: in the right panel, **Session options → Accelerator → GPU T4 x2** and **Internet → On**.

Each run = one 'Save Version → Save & Run All (Commit)'. Change the Settings cell first. Results appear in the **Output** tab of that version (folder `runs/<run name>`).

In [ ]:
!nvidia-smi   # should list Tesla T4 (two of them is fine, we only use the first)

In [ ]:
# get the code (kept outside /kaggle/working so it is not saved as output)
%cd /tmp
!rm -rf repo
!git clone https://github.com/phoebeyap04/dsa4212-small-llm.git repo
%cd /tmp/repo
!git log --oneline -1
!pip install -q tiktoken datasets tokenizers

## Data
Builds the dataset from scratch (~20 min). Kaggle has no Google Drive, so this runs every time.

In [ ]:
%cd /tmp/repo/data/tinystories
!python prepare.py
!cat manifest.txt
%cd /tmp/repo

## Settings
Change these before each run. Keep budget = 3600.0 for real runs.

In [ ]:
size = 'large'    # small, medium or large
pool = 'large'    # small or large
seed = 1337       # 1337 or 2024
budget = 3600.0

run_name = f'{size}_{pool}_seed{seed}'
if budget != 3600.0:
    run_name = run_name + '_test'
out = f'/kaggle/working/runs/{run_name}'
print(out)

## Train

In [ ]:
!python train.py config/ts_{size}.py --train_file=train_{pool}.bin --seed={seed} --time_budget={budget} --out_dir={out}

## Results
ckpt.pt is deleted at the end (only needed for resuming) to keep the download small.

In [ ]:
!cat {out}/summary.json
!python sample.py --out_dir={out} --num_samples=2 --max_new_tokens=200
!rm -f {out}/ckpt.pt
!ls -lh {out}